In [8]:
import torch
from torch import nn
import numpy as np
from string import Template
import math

import os 
import sys
sys.path.append(os.path.abspath("./"))
from input_image_gen import generate_input_file
from conv_layer_generator import *
from dense_layer_generator import *
from codebooks_defs_generator import *

In [9]:
def conv_out_size(in_size, stride, padding, k_size):
    return int(((in_size - k_size + (2 * padding)) / stride) + 1) 

def max_pool_out_size(in_size, pool_size):
    return int(in_size / pool_size)

In [10]:
# My test settings (MVP)
N_LEARNERS = 1
CODEBOOK_SIZE = 8

IN_CHANNELS = 1
IN_HEIGHT = 4
IN_WIDTH = IN_HEIGHT

TILE_L2_SIZE = 0
TILE_L1_SIZE = 0

SVE_LANES = 1

SAME_SEQ = True     # Specifies if the learners should have the same access sequence

In [11]:

# N_LEARNERS = 4
# CODEBOOK_SIZE = 8

# IN_CHANNELS = 1
# IN_HEIGHT = 28
# IN_WIDTH = IN_HEIGHT

# TILE_L2_SIZE = 6
# TILE_L1_SIZE = 9

# SVE_LANES = 4

# SAME_SEQ = True     # Specifies if the learners should have the same access sequence

conv_0 = {"type": "conv",
          "in_ch": IN_CHANNELS, 
          "out_ch": 6,
          "k_size": 5, 
          "stride": 1, 
          "padding": 0}

max_pool_0 = {"type": "maxpool",
              "size": 2}

conv_1 = {"type": "conv",
          "in_ch": conv_0["out_ch"], 
          "out_ch": 16,
          "k_size": 5,
          "stride": 1,
          "padding": 0}

max_pool_1 = {"type": "maxpool",
              "size": 2}

dense_2 = {"type": "dense",
           "out_size": 120}

dense_3 = {"type": "dense",
           "out_size": 84}

dense_4 = {"type": "dense",
           "out_size": 10}

dense_test = {"type": "dense", "out_size": 4}

NN_structure = [conv_0, max_pool_0, conv_1, max_pool_1, dense_2, dense_3, dense_4]
# NN_structure = [conv_test]
NN_structure = [dense_test]


In [12]:
USE_BIAS = False

USE_F16 = False
USE_CODEBOOKS = True   # or False if you want NO_CODEBOOKS


# OUT_FOLDER = "./generated_headers/"
OUT_FOLDER = "./../lenet_definitions/"

# generate_cb_definitions(OUT_FOLDER + "codebooks_def.h", N_LEARNERS, CODEBOOK_SIZE, SVE_LANES) 
generate_cb_definitions(OUT_FOLDER + "codebooks_def.h", N_LEARNERS, CODEBOOK_SIZE, SVE_LANES, USE_BIAS, USE_F16, SAME_SEQ, USE_CODEBOOKS) # Fix missing parameters in the function call
# input_values = generate_input_file(OUT_FOLDER + "input_image.h", IN_CHANNELS, IN_HEIGHT, IN_WIDTH)
input_values = generate_input_file(OUT_FOLDER + "input_image.h", IN_CHANNELS, IN_HEIGHT, IN_WIDTH, USE_F16) # Fix missing parameters in the function call

# These are in case the layer is a conv or max pool
input_ch = IN_CHANNELS
input_height = IN_HEIGHT
input_width = IN_WIDTH

# This is in case the layer is a dense layer
in_size = IN_CHANNELS * IN_HEIGHT * IN_WIDTH


# This list is to save all the kernel values, so to test with torch
kernels = []

# This list is to save all the dense layer values, so   to test with torch
dense_weights = []

# Final torch network, one per learner
network = [nn.ModuleDict({}) for _ in range(N_LEARNERS)] # Create a list of ModuleDicts, one for each learner, to hold the layers of each learner's network
# network = [
#     ModuleDict({}),
#     ModuleDict({}),
#     ModuleDict({})
# ]

for lay_cnt, layer in enumerate(NN_structure):
    print("[{}] {}".format(lay_cnt, layer["type"]))
    print("\t", layer)

    if layer['type'] == "conv":
        in_shape = (layer["in_ch"], input_height, input_width)
        out_channels = layer["out_ch"]
        out_height = conv_out_size(input_height, layer["stride"], layer["padding"], layer["k_size"])
        out_width = conv_out_size(input_width, layer["stride"], layer["padding"], layer["k_size"])
        out_shape = (out_channels, out_height, out_width)

        # Get the codebooks values for the learners and generate the header
        # cb_values, k_values, cb_biases_values, biases_values = generate_kernel_header_file(SAME_SEQ, OUT_FOLDER + "conv_header_{}.h".format(lay_cnt), lay_cnt, N_LEARNERS, CODEBOOK_SIZE, out_channels, layer["in_ch"], layer["k_size"], layer['stride'], layer["padding"], TILE_L2_SIZE, TILE_L1_SIZE) 
        cb_values, k_values, biases_values = generate_kernel_header_file(SAME_SEQ, OUT_FOLDER + "conv_header_{}.h".format(lay_cnt), lay_cnt, N_LEARNERS, CODEBOOK_SIZE, out_channels, layer["in_ch"], layer["k_size"], layer['stride'], layer["padding"], TILE_L2_SIZE, TILE_L1_SIZE, USE_F16, USE_CODEBOOKS) # Fix missing parameters in the function call and remove cb_biases_values since it's not used
        kernels.append(k_values)
        

        # Per each learner network, force the kernel values and append the layer to the learner network
        for learner in range(N_LEARNERS):

            new_conv = nn.Conv2d(in_channels=layer['in_ch'],
                                                        out_channels=layer["out_ch"],
                                                        kernel_size=(layer["k_size"], layer["k_size"]),
                                                        stride = layer['stride'],
                                                        padding=layer["padding"],
                                                        bias = USE_BIAS)
            with torch.no_grad():
                new_conv.weight.copy_(torch.tensor(k_values[learner]).view(layer["out_ch"], layer['in_ch'], layer["k_size"], layer["k_size"]))

                if USE_BIAS:
                    new_conv.bias.copy_(torch.tensor(biases_values[learner]))

            network[learner]["conv{}".format(lay_cnt)] = new_conv


        input_ch = out_channels
        input_height = out_height
        input_width = out_width

        in_size = out_channels * out_height * out_width


    elif layer["type"] == "maxpool":
        in_shape = (input_ch, input_height, input_width)
        out_channels = input_ch
        out_height = max_pool_out_size(input_height, layer["size"])
        out_width = max_pool_out_size(input_width, layer["size"])
        out_shape = (out_channels, out_height, out_width)

        # Per each learner network, append the layer to the network
        for learner in range(N_LEARNERS):
            new_maxpool = nn.MaxPool2d(layer["size"])
            network[learner]["maxpool{}".format(lay_cnt)] = new_maxpool

        input_ch = out_channels
        input_height = out_height
        input_width = out_width

        in_size = out_channels * out_height * out_width

    elif layer["type"] == "dense":
        in_shape = in_size
        out_size = layer["out_size"]

        # dense_values = generate_template_dense(SAME_SEQ, OUT_FOLDER + "dense_header_{}.h".format(lay_cnt), lay_cnt, N_LEARNERS, CODEBOOK_SIZE, 5, in_size, out_size)
        dense_values, dense_biases_values = generate_template_dense(SAME_SEQ, OUT_FOLDER + "dense_header_{}.h".format(lay_cnt), lay_cnt, N_LEARNERS, CODEBOOK_SIZE, 5, in_size, out_size, USE_F16, USE_CODEBOOKS) # Fix missing parameters in the function call
        dense_weights.append(dense_values)

        # for learner in range(N_LEARNERS):

        #     new_dense = nn.Linear(in_shape, layer["out_size"], bias=USE_BIAS)
            
        #     with torch.no_grad():
        #         new_dense.weight.copy_(torch.Tensor(dense_values[learner]).view(out_size, in_shape))

        #     network[learner]["dense{}".format(lay_cnt)] = new_dense

        for learner in range(N_LEARNERS):

            new_dense = nn.Linear(in_shape, layer["out_size"], bias=USE_BIAS)
            
            with torch.no_grad():
                new_dense.weight.copy_(torch.Tensor(dense_values[learner]).view(out_size, in_shape))

                if USE_BIAS:
                    new_dense.bias.copy_(torch.tensor(dense_biases_values[learner]))

            network[learner]["dense{}".format(lay_cnt)] = new_dense

        out_shape = out_size

        in_size = out_shape
        
    else:
        print("ERROR!")
        exit(1)



    print("In shape:", in_shape)
    print("Out shape:", out_shape)
    print()
    print("First 10 input values:")
    print(input_values[:10])

[0] dense
	 {'type': 'dense', 'out_size': 4}
In shape: 16
Out shape: 4

First 10 input values:
[-0.250919762305275, 0.9014286128198323, 0.4639878836228102, 0.1973169683940732, -0.687962719115127, -0.6880109593275947, -0.8838327756636011, 0.7323522915498704, 0.2022300234864176, 0.416145155592091]


In [13]:
# print(len(cb_biases_values))

# for cb in cb_biases_values:
#     print(cb)

In [14]:
# print(len(biases_values))

# for bv in biases_values:
#     print(len(bv))
#     print(bv)

In [15]:
print(len(dense_biases_values))

for bv in dense_biases_values:
    print(len(bv))
    print(bv)

1
4
[0.27381802 0.47228079 0.38875052 0.26210623]


In [16]:
input = torch.tensor(input_values, dtype=torch.float32).view(-1)   # flatten
print("input shape:", input.shape)  # should be [in_size]

for ens in range(N_LEARNERS):
    print(f"\n=============== LEARNER {ens} ===============\n")

    y = network[ens]["dense0"](input)     # only layer
    print("y shape:", y.shape)
    print("y:", y)

# input = torch.Tensor(input_values).view(IN_CHANNELS, IN_HEIGHT, IN_WIDTH)

# print(input.shape)

# for ens in range(N_LEARNERS):
#     print("\n=============== LEARNER {} ===============\n".format(ens))
#     x = network[ens]['conv0'](input)
#     print(x.shape)
#     # print(x[0])
#     # break

#     x = torch.relu(x)
#     # print(x.shape)
#     # print(x)

#     x = network[ens]['maxpool1'](x)
#     print(x.shape)
#     # print(x)
#     # break

#     x = network[ens]['conv2'](x)
#     print(x.shape)
#     # print(x[0])
#     # break

#     x = torch.relu(x)
#     print(x.shape)
#     # print(x)
#     # break

#     x = network[ens]['maxpool3'](x)
#     print(x.shape)
#     # print(x)
#     # break

#     x = x.flatten()
#     print(x.shape)
#     # print(x)

#     x = network[ens]['dense4'](x)
#     # print(x.shape)
#     # print(x)
#     # break

#     x = torch.relu(x)
#     print(x.shape)
#     # print(x)
#     # break

#     x = network[ens]['dense5'](x)
#     # print(x.shape)
#     # print(x)

#     x = torch.relu(x)
#     print(x.shape)
#     # print(x)
#     # break

#     x = network[ens]['dense6'](x)
#     print(x.shape)
#     print(x)

#     # break

input shape: torch.Size([16])

=============== LEARNER 0 ===============

y shape: torch.Size([4])
y: tensor([ 0.2270, -0.2760,  0.1021,  0.0129], grad_fn=<SqueezeBackward4>)


In [17]:
print(network[0]["dense0"].bias)   # should be None if bias=False

None


Yes. For matching your notebook values, set bias to zero in this C test.

Change this call in [dense_only_lenet.c:21](/home/jerry/NN_layers-Jerry/Full_NN/dense_only_lenet.c:21):

```c
exec_compact(dense_0, input_flat, weight_idx_compact_0, codebooks_0[0], bias_0[0], out);
```

to:

```c
float zero_bias[OUTPUT_SIZE_0] = {0};
exec_compact(dense_0, input_flat, weight_idx_compact_0, codebooks_0[0], zero_bias, out);
```

Do not “comment out” `bias_0` in the header if you still want full-model runs; just override it locally in this debug file.

What bias is:
- In a dense layer, output is `y = W*x + b`.
- `b` (bias) is one learned offset per output neuron.
- It comes from training (same as weights), then exported into your generated header (`bias_0` in [dense_header_0.h:61](/home/jerry/NN_layers-Jerry/Full_NN/lenet_definitions/dense_header_0.h:61)).

Why notebook may not include it:
- You likely compared against only `W*x` (or used a layer/config with `bias=False`, or forgot to add bias in manual calc).
- Your numbers already confirm this: notebook matches dot-product-only; C matches dot+bias.

If you want, I can patch `dense_only_lenet.c` to print both:
1. `W*x` (zero bias), and  
2. `W*x+b` (with `bias_0[0]`)  
side by side.